In [1]:
import pandas as pd
import numpy as np
import os
import torch

from Model.Query2Label import Query2Label

# model_dir = "/mnt/d/ML/Kaggle/CAFA6-new/downloads/test/"

In [2]:
# Load ESM 8M model from Hugging Face
from transformers import AutoTokenizer, AutoModel
import torch

# Load model and tokenizer (ESM2 8M parameters)
model_name = "facebook/esm2_t6_8M_UR50D"
print(f"Loading {model_name}...")

esm_tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
esm_model = esm_model.to(device)
esm_model.eval()

print(f"Model loaded successfully on {device}!")
print(f"Model has {sum(p.numel() for p in esm_model.parameters()):,} parameters")


Loading facebook/esm2_t6_8M_UR50D...


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully on cuda!
Model has 7,511,801 parameters


In [3]:

# Example forward pass
example_sequences = [
    "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEK",
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTG"
]

print(f"\nRunning forward pass on {len(example_sequences)} example sequences...")

# Tokenize sequences
inputs = esm_tokenizer(example_sequences, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

print(f"Input shape: {inputs['input_ids'].shape}")

# Forward pass
with torch.no_grad():
    outputs = esm_model(**inputs)

# Get embeddings
last_hidden_states = outputs.last_hidden_state  # (batch_size, seq_len, hidden_dim)
pooled_output = last_hidden_states.mean(dim=1)  # Average pooling: (batch_size, hidden_dim)

print(f"\nOutput shapes:")
print(f"  Last hidden states: {last_hidden_states.shape}")
print(f"  Pooled output: {pooled_output.shape}")
print(f"\nExample embedding stats:")
print(f"  Mean: {pooled_output[0].mean().item():.4f}")
print(f"  Std: {pooled_output[0].std().item():.4f}")
print(f"  Min: {pooled_output[0].min().item():.4f}")
print(f"  Max: {pooled_output[0].max().item():.4f}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Running forward pass on 2 example sequences...
Input shape: torch.Size([2, 58])

Output shapes:
  Last hidden states: torch.Size([2, 58, 320])
  Pooled output: torch.Size([2, 320])

Example embedding stats:
  Mean: -0.0097
  Std: 0.3247
  Min: -5.2045
  Max: 0.4490


In [4]:
inputs['input_ids'].shape

torch.Size([2, 58])

In [11]:
query_embeds = torch.randn(2, 64, 512).to(device)
query_embeds.shape

torch.Size([2, 64, 512])

In [8]:
model = Query2Label(
    esm_model,
    feature_encoder_output_dim=320,
    num_classes = 64,
    in_dim = 512,
    num_encoder_layers  = 1,
    num_decoder_layers = 2,
    use_positional_encoding = False
)

model = model.to(device)

In [12]:
outputs = model(query_embeds, inputs)

In [14]:
outputs.shape

torch.Size([2, 64])